# Blood Bridge — ML Supply Depletion & Demand Forecasting Engine
### IBM Z Datathon 2026 · AI Secured Track (Innovation without Exposure)

This Google Colab notebook trains the **Machine Learning Supply Depletion and Demand Forecasting Model** for the Blood Bridge platform.

**Core Objectives:**
1. **Calculate Average Usage per Day and Weekly Usage** across all blood groups and components.
2. **Predict Time to Depletion (Hours/Days until Stockout)** for every blood bank.
3. **Triage Stockout Risks** into operational tiers (`CRITICAL < 24h`, `HIGH < 72h`, `MODERATE < 7d`, `STABLE > 7d`).
4. **Strict Zero-Exposure Privacy**: Operates on anonymized blood inventory and request volume with zero patient or donor identifiers.

## Step 1: Install Dependencies

In [ ]:
# Install dependencies in Google Colab
!pip install -q lightgbm scikit-learn pandas numpy matplotlib seaborn joblib

## Step 2: Import Libraries & Configure Environment

In [ ]:
import os
import math
import json
import random
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, date, timedelta, timezone

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
import lightgbm as lgb

sns.set_theme(style="whitegrid")
print("Libraries imported successfully!")

## Step 3: Multi-Source Dataset Loader (Local / GitHub / In-Memory Generator)
Searches for local dataset files, falls back to GitHub raw URL, and if private/unreachable, automatically generates the synthetic demonstration dataset directly in memory.

In [ ]:
# Multi-source dataset loader: Local -> GitHub -> In-Memory Generator Fallback
import os
import random
from datetime import date, datetime, timedelta

# 1. Search candidate directories
candidate_dirs = [
    "datasets",
    "../datasets",
    "Blood-Bridge/datasets",
    "/content/Blood-Bridge/datasets",
    "backend/datasets",
    "database/datasets"
]
found_dir = None
for d in candidate_dirs:
    if os.path.exists(os.path.join(d, "blood_inventory.csv")):
        found_dir = d
        break

inv_df, req_df, banks_df, hosp_df = None, None, None, None

if found_dir:
    print(f"[+] Found local datasets directory at: {found_dir}")
    inv_df = pd.read_csv(os.path.join(found_dir, "blood_inventory.csv"))
    req_df = pd.read_csv(os.path.join(found_dir, "blood_requests.csv"))
    banks_df = pd.read_csv(os.path.join(found_dir, "blood_banks.csv"))
    hosp_df = pd.read_csv(os.path.join(found_dir, "hospitals.csv"))
else:
    # 2. Try fetching from GitHub raw URL
    repo_url = "https://raw.githubusercontent.com/TBxHavoc/Blood-Bridge/database/datasets/"
    try:
        inv_df = pd.read_csv(repo_url + "blood_inventory.csv")
        req_df = pd.read_csv(repo_url + "blood_requests.csv")
        banks_df = pd.read_csv(repo_url + "blood_banks.csv")
        hosp_df = pd.read_csv(repo_url + "hospitals.csv")
        print(f"[+] Successfully loaded datasets from GitHub repository!")
    except Exception as e:
        # 3. Autonomous Inlined Generator Fallback (Runs 100% self-contained in Google Colab)
        print(f"[i] Remote GitHub not reachable without auth. Initializing in-memory synthetic dataset generator...")
        random.seed(42)
        
        CITIES = [
            ("Mumbai", 19.0760, 72.8777), ("Delhi", 28.6139, 77.2090), ("Bengaluru", 12.9716, 77.5946),
            ("Chennai", 13.0827, 80.2707), ("Pune", 18.5204, 73.8567), ("Hyderabad", 17.3850, 78.4867),
            ("Kolkata", 22.5726, 88.3639), ("Ahmedabad", 23.0225, 72.5714), ("Jaipur", 26.9124, 75.7873),
            ("Lucknow", 26.8467, 80.9462)
        ]
        BG_WEIGHTS = {"O+": 0.35, "A+": 0.25, "B+": 0.15, "AB+": 0.05, "O-": 0.07, "A-": 0.05, "B-": 0.05, "AB-": 0.03}
        
        # Generate Blood Banks
        banks_list = []
        for i in range(20):
            city, lat, lon = CITIES[i % len(CITIES)]
            banks_list.append({
                "code": f"BB_{i+1:03d}", "name": f"{city} Blood Bank {i+1}", "city": city,
                "latitude": lat, "longitude": lon, "capacity_units": random.randint(300, 1200), "is_active": True
            })
        banks_df = pd.DataFrame(banks_list)
        
        # Generate Hospitals
        hosp_list = []
        for i in range(10):
            city, lat, lon = CITIES[i % len(CITIES)]
            hosp_list.append({
                "code": f"H_{i+1:03d}", "name": f"{city} General Hospital {i+1}", "city": city,
                "latitude": lat, "longitude": lon, "emergency_dept_capacity": random.choice(["small", "medium", "large"]),
                "is_active": True
            })
        hosp_df = pd.DataFrame(hosp_list)
        
        # Generate Inventory
        today_dt = date.today()
        inv_list = []
        inv_id = 0
        for b in banks_list:
            for _ in range(random.randint(5, 8)):
                inv_id += 1
                comp = "RBC" if random.random() < 0.65 else "WHOLE_BLOOD"
                shelf = 42 if comp == "RBC" else 35
                coll = today_dt - timedelta(days=random.randint(2, shelf - 5))
                exp = coll + timedelta(days=shelf)
                avail = random.randint(4, 28)
                res = random.choice([0, 0, 1, 2]) if avail > 5 else 0
                bg_choice = random.choices(list(BG_WEIGHTS.keys()), weights=list(BG_WEIGHTS.values()), k=1)[0]
                inv_list.append({
                    "code": f"INV_{inv_id:03d}", "blood_bank_code": b["code"], "blood_group": bg_choice,
                    "component_type": comp, "units_available": avail, "units_reserved": res,
                    "collection_date": str(coll), "expiry_date": str(exp), "is_available": True
                })
        inv_df = pd.DataFrame(inv_list)
        
        # Generate Requests
        now_dt = datetime.now(timezone.utc).replace(tzinfo=None)
        req_list = []
        for i in range(45):
            h = random.choice(hosp_list)
            sub_time = now_dt - timedelta(hours=random.randint(1, 48))
            bg_choice = random.choices(list(BG_WEIGHTS.keys()), weights=list(BG_WEIGHTS.values()), k=1)[0]
            req_list.append({
                "code": f"REQ_SEED_{i+1:03d}", "hospital_code": h["code"], "blood_group": bg_choice,
                "component_type": "RBC" if random.random() < 0.65 else "WHOLE_BLOOD",
                "quantity_units": random.randint(1, 6), "urgency": random.randint(1, 5),
                "status": random.choice(["OPEN", "FULFILLED", "FULFILLED", "CANCELLED"]),
                "request_timestamp": str(sub_time)
            })
        req_df = pd.DataFrame(req_list)
        
        # Save to local datasets folder in Colab
        os.makedirs("datasets", exist_ok=True)
        inv_df.to_csv("datasets/blood_inventory.csv", index=False)
        req_df.to_csv("datasets/blood_requests.csv", index=False)
        banks_df.to_csv("datasets/blood_banks.csv", index=False)
        hosp_df.to_csv("datasets/hospitals.csv", index=False)
        print(f"[+] Fallback active: Generated {len(inv_df)} inventory items, {len(req_df)} requests, and saved to ./datasets/")

print(f"\nDataset Ready: {len(inv_df)} inventory rows | {len(req_df)} clinical requests | {len(banks_df)} blood banks")
inv_df.head(5)

## Step 4: Calculate Daily & Weekly Usage Statistics
Computes empirical consumption velocity per Blood Group and Component from historical clinical orders.

In [ ]:
# Filter out cancelled requests
valid_reqs = req_df[req_df["status"] != "CANCELLED"].copy()
valid_reqs["request_timestamp"] = pd.to_datetime(valid_reqs["request_timestamp"])

# Observation time horizon in days
t_min = valid_reqs["request_timestamp"].min()
t_max = valid_reqs["request_timestamp"].max()
elapsed_days = max(1.0, (t_max - t_min).total_seconds() / 86400.0)

# Normalize component names
valid_reqs["component_type"] = valid_reqs["component_type"].apply(
    lambda x: "RBC" if "RBC" in str(x).upper() else "WHOLE_BLOOD"
)

# Group and calculate usage metrics
usage_summary = (
    valid_reqs.groupby(["blood_group", "component_type"])
    .agg(
        total_units=("quantity_units", "sum"),
        request_count=("code", "count"),
        avg_urgency=("urgency", "mean")
    )
    .reset_index()
)

usage_summary["avg_usage_per_day"] = (usage_summary["total_units"] / elapsed_days).round(2)
usage_summary["weekly_usage"] = (usage_summary["avg_usage_per_day"] * 7.0).round(2)
usage_summary["hourly_burn_rate"] = (usage_summary["avg_usage_per_day"] / 24.0).round(4)

print("=" * 70)
print("HISTORICAL USAGE STATISTICS (DAILY & WEEKLY)")
print("=" * 70)
display(usage_summary.sort_values("avg_usage_per_day", ascending=False))

# Visualize Usage Rates
plt.figure(figsize=(10, 5))
sns.barplot(data=usage_summary, x="blood_group", y="avg_usage_per_day", hue="component_type", palette="viridis")
plt.title("Average Clinical Usage Per Day by Blood Group & Component", fontsize=14, fontweight="bold")
plt.ylabel("Units Per Day")
plt.xlabel("Blood Group")
plt.tight_layout()
plt.show()

## Step 5: Feature Engineering for Depletion Forecasting
Combines current stock, unreserved units, batch expiry attrition, and burn rates into training vectors.

In [ ]:
# Preprocess Inventory
inv = inv_df[inv_df["is_available"] == True].copy()
inv["component_type"] = inv["component_type"].apply(
    lambda x: "RBC" if "RBC" in str(x).upper() else "WHOLE_BLOOD"
)
inv["unreserved_units"] = np.maximum(0, inv["units_available"] - inv["units_reserved"])
inv["expiry_date"] = pd.to_datetime(inv["expiry_date"]).dt.date

# Aggregate on-hand supply per blood bank and blood group
bank_inv = (
    inv.groupby(["blood_bank_code", "blood_group", "component_type"])
    .agg(
        total_available=("units_available", "sum"),
        total_reserved=("units_reserved", "sum"),
        total_unreserved=("unreserved_units", "sum"),
        earliest_expiry=("expiry_date", "min"),
        batch_count=("code", "count")
    )
    .reset_index()
)

# Merge with demand metrics and blood bank capacity
df = pd.merge(bank_inv, usage_summary, on=["blood_group", "component_type"], how="left")
df["avg_usage_per_day"] = df["avg_usage_per_day"].fillna(0.1)
df["weekly_usage"] = df["weekly_usage"].fillna(0.7)
df["hourly_burn_rate"] = df["hourly_burn_rate"].fillna(0.004)
df["avg_urgency"] = df["avg_urgency"].fillna(2.5)

df = pd.merge(df, banks_df[["code", "city", "capacity_units"]], left_on="blood_bank_code", right_on="code", how="left")

today = date.today()
df["days_to_expiry"] = df["earliest_expiry"].apply(lambda d: max(0.0, (d - today).days) if pd.notnull(d) else 30.0)

# Apportion burn rate across the network
num_banks = len(banks_df["code"].unique())
df["bank_daily_burn"] = np.maximum(0.1, df["avg_usage_per_day"] / num_banks)
df["bank_hourly_burn"] = df["bank_daily_burn"] / 24.0

# Compute Target Hours to Depletion and Risk Tier
hours_list = []
tier_list = []
label_list = []

for _, row in df.iterrows():
    stock = row["total_unreserved"]
    burn = row["bank_hourly_burn"]
    exp_h = row["days_to_expiry"] * 24.0
    
    if stock <= 0:
        h = 0.0
        tier = 0
        lbl = "IMMEDIATE_EMPTY"
    else:
        h = min(stock / burn, exp_h)
        if h <= 24.0:
            tier = 1
            lbl = "CRITICAL"
        elif h <= 72.0:
            tier = 2
            lbl = "HIGH"
        elif h <= 168.0:
            tier = 3
            lbl = "MODERATE"
        else:
            tier = 4
            lbl = "STABLE"
            
    hours_list.append(round(h, 2))
    tier_list.append(tier)
    label_list.append(lbl)

df["target_hours"] = hours_list
df["target_tier"] = tier_list
df["target_label"] = label_list
print(f"Feature matrix prepared with {len(df)} samples!")
df[["blood_bank_code", "blood_group", "total_unreserved", "bank_daily_burn", "target_hours", "target_label"]].head(8)

## Step 6: Train Model 1 — LightGBM Depletion Regressor
Predicts continuous hours and days until blood supplies are exhausted.

In [ ]:
# Encode categorical variables
le_bg = LabelEncoder()
le_ct = LabelEncoder()
le_city = LabelEncoder()

df["feat_bg"] = le_bg.fit_transform(df["blood_group"])
df["feat_ct"] = le_ct.fit_transform(df["component_type"])
df["feat_city"] = le_city.fit_transform(df["city"].fillna("Unknown"))

feature_cols = [
    "feat_bg", "feat_ct", "feat_city",
    "total_unreserved", "total_available", "total_reserved",
    "days_to_expiry", "bank_daily_burn", "capacity_units", "avg_urgency"
]

X = df[feature_cols].fillna(0)
y_reg = df["target_hours"]
y_clf = df["target_tier"]

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

# Train LightGBM Regressor
reg = lgb.LGBMRegressor(
    n_estimators=120,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)
reg.fit(X_train, y_reg_train)
reg_preds = reg.predict(X_test)

mae = mean_absolute_error(y_reg_test, reg_preds)
r2 = r2_score(y_reg_test, reg_preds)

print("=" * 60)
print("REGRESSION MODEL PERFORMANCE (HOURS TO DEPLETION)")
print("=" * 60)
print(f"R-squared (R2 Score):        {r2:.4f}")
print(f"Mean Absolute Error (MAE):    {mae:.2f} hours (~{mae/24.0:.2f} days)")

# Plot Actual vs Predicted
plt.figure(figsize=(7, 6))
plt.scatter(y_reg_test, reg_preds, color="#1f77b4", alpha=0.8, edgecolors="k")
plt.plot([y_reg_test.min(), y_reg_test.max()], [y_reg_test.min(), y_reg_test.max()], "r--", lw=2)
plt.xlabel("Ground Truth Depletion (Hours)")
plt.ylabel("Model Predicted Depletion (Hours)")
plt.title(f"LightGBM: Actual vs. Predicted Depletion Hours (R2 = {r2:.3f})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 7: Train Model 2 — Stockout Risk Classifier
Predicts categorical triage risk tiers (`CRITICAL`, `HIGH`, `MODERATE`, `STABLE`).

In [ ]:
# Train Random Forest Classifier
clf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
clf.fit(X_train, y_clf_train)
clf_preds = clf.predict(X_test)

acc = accuracy_score(y_clf_test, clf_preds)
print("=" * 60)
print(f"CLASSIFICATION ACCURACY: {acc * 100:.2f}%")
print("=" * 60)

# Feature Importance Plot
importances = pd.Series(reg.feature_importances_, index=feature_cols).sort_values(ascending=True)
plt.figure(figsize=(9, 5))
importances.plot(kind="barh", color="#2ca02c")
plt.title("Explainable AI: Feature Importances for Supply Depletion", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

## Step 8: Save Model Artifacts
Exports the trained models and metadata for loading in FastAPI or offline inference.

In [ ]:
os.makedirs("models", exist_ok=True)
artifact_bundle = {
    "regressor": reg,
    "classifier": clf,
    "feature_cols": feature_cols,
    "encoders": {"bg": le_bg, "ct": le_ct, "city": le_city},
    "usage_summary": usage_summary.to_dict(orient="records"),
    "model_type": "LightGBM + Random Forest Hybrid",
    "timestamp": datetime.now(timezone.utc).isoformat()
}

output_file = "models/depletion_model.pkl"
with open(output_file, "wb") as f:
    pickle.dump(artifact_bundle, f)

print(f"Model artifact successfully exported to: {output_file} ({os.path.getsize(output_file)/1024:.1f} KB)")

## Step 9: Sample Inference Demo

In [ ]:
# Interactive Inference Function
def forecast_supply_line(blood_bank_code, blood_group, component_type, unreserved_units, days_to_expiry, city="Mumbai"):
    with open("models/depletion_model.pkl", "rb") as f:
        bundle = pickle.load(f)
        
    m_reg = bundle["regressor"]
    m_clf = bundle["classifier"]
    
    bg_idx = bundle["encoders"]["bg"].transform([blood_group])[0]
    ct_idx = bundle["encoders"]["ct"].transform([component_type])[0]
    city_idx = bundle["encoders"]["city"].transform([city])[0]
    
    # Get daily burn from usage summary
    burn = 0.5
    for u in bundle["usage_summary"]:
        if u["blood_group"] == blood_group and u["component_type"] == component_type:
            burn = u["avg_usage_per_day"] / 20.0
            break
            
    sample_vec = pd.DataFrame([{
        "feat_bg": bg_idx,
        "feat_ct": ct_idx,
        "feat_city": city_idx,
        "total_unreserved": unreserved_units,
        "total_available": unreserved_units,
        "total_reserved": 0,
        "days_to_expiry": days_to_expiry,
        "bank_daily_burn": burn,
        "capacity_units": 500,
        "avg_urgency": 3.0
    }])[bundle["feature_cols"]]
    
    pred_hours = m_reg.predict(sample_vec)[0]
    pred_tier_idx = m_clf.predict(sample_vec)[0]
    
    tier_names = {0: "IMMEDIATE_EMPTY", 1: "CRITICAL (<24h)", 2: "HIGH (1-3 days)", 3: "MODERATE (3-7 days)", 4: "STABLE (>7 days)"}
    
    print("=" * 65)
    print(f"INFERENCE PREDICTION: {blood_bank_code} | {blood_group} {component_type}")
    print("=" * 65)
    print(f"Unreserved Stock:           {unreserved_units} units")
    print(f"Days until Batch Expiry:    {days_to_expiry} days")
    print(f"Predicted Depletion Time:   {pred_hours:.1f} hours ({pred_hours/24.0:.1f} days)")
    print(f"Assigned Risk Tier:         {tier_names.get(pred_tier_idx, 'UNKNOWN')}")
    print("=" * 65)

# Test Case 1: Critical Emergency (Low Stock)
forecast_supply_line("BB_001", "O+", "RBC", unreserved_units=3, days_to_expiry=15)

# Test Case 2: Stable Supply
forecast_supply_line("BB_002", "A+", "WHOLE_BLOOD", unreserved_units=25, days_to_expiry=25)
